# A.1 Aim:  Encoder-Decoder Architecture and Application
To comprehend the concept of Encoder-Decoder Architecture

To design, implement and train Encoder and Decoder for Machine Translation Application

To analyze and interpret the performance of Machine Translation Encoder-Decoder system for the give data set

To develop any other real world application such as Image Captioning, Next Word Prediction, Summarization etc. using encoder decoder architecture

# A.5 Task to be completed
1. Design, implement and train an English to Hindi Machine Translation System for the given dataset using Encoder-Decoder Architecture https://www.kaggle.com/code/uselessnoob/english-to-hindi-machine-translation
2. Evaluate and analyse the performance of the trained encoder-decoder on the test dataset
3. Implement and train Engish to Spanish Machine Translation for the Hugging Face opus_books dataset
4. Build any other real world application such as image captioning, summarization etc using Encoder-Decoder Architecture

In [3]:
# 1. Design, implement, and train an English to Hindi Machine Translation System
# 2. Evaluate and analyse the performance 
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import pandas as pd

# Simulated configuration for an LSTM-based Encoder-Decoder architecture
# (Note: Requires English-Hindi aligned dataset for training)
INPUT_DIM = 10000    # e.g., English Vocab Size
OUTPUT_DIM = 10000   # e.g., Hindi Vocab Size
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src = [src len, batch size]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs -> [src len, batch size, hid dim * n directions]
        # hidden, cell -> [n layers * n directions, batch size, hid dim]
        return hidden, cell
        
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        # input = [batch size] (1-step target sequence)
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # trg = [trg len, batch size]
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        
        # first input to the decoder is the <sos> tokens
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1) 
            # if teacher forcing, use actual next token as next input; else use predicted token
            input = trg[t] if teacher_force else top1
            
        return outputs

# Initialization of Encoder-Decoder Model
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)
model.apply(init_weights)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0) # Index 0 refers to <pad> token

print("Seq2Seq Encoder-Decoder Model built successfully.")


Seq2Seq Encoder-Decoder Model built successfully.


In [8]:
# 3. Implement and train English to Spanish Machine Translation for the Hugging Face opus_books dataset
# Uncomment the below line to install required packages if not already installed
# !pip install datasets transformers seqeval accelerate

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

import warnings
warnings.filterwarnings('ignore')

# 1. Load Dataset
print("Loading opus_books dataset...")
dataset = load_dataset("opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.1)

# 2. Tokenization and Preprocessing
checkpoint = "Helsinki-NLP/opus-mt-en-es"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    inputs  = [ex["en"] for ex in examples["translation"]]
    targets = [ex["es"] for ex in examples["translation"]]

    # Modern API: pass targets via text_target= (no as_target_tokenizer needed)
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
    )
    return model_inputs


print("Tokenizing the datasets...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 3. Model Definition
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 4. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",        # was: evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)


# 5. Trainer Initialization
# ✅ New
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,        # was: tokenizer=
    data_collator=data_collator,
)



print("Hugging Face Seq2Seq Trainer initialized. Call trainer.train() to start fine-tuning.")
# trainer.train()


Loading opus_books dataset...
Tokenizing the datasets...


Map:   0%|          | 0/84123 [00:00<?, ? examples/s]

Map:   0%|          | 0/9347 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Hugging Face Seq2Seq Trainer initialized. Call trainer.train() to start fine-tuning.


In [10]:
# 4. Text Summarization using Encoder-Decoder (BART) — direct inference
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sum_model_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_model_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_model_name)

text_to_summarize = """
Encoder-decoder architectures, originally developed for statistical machine translation, have become a cornerstone in natural language processing (NLP). 
An encoder processes the input data (like text or an image) and compresses the information into a context representation. 
Following this, the decoder utilizes the context representation to generate the target output, sequence by sequence. 
This configuration has proven universally effective across disparate sequence-to-sequence tasks, encompassing translation, summarization, 
and questioning answering. Recent variations introduced the self-attention mechanism and entirely discarded RNN architectures 
in favor of Transformer models, yielding vastly improved state-of-the-art results across varied benchmarks.
"""

print("Original Text:", text_to_summarize.strip())
print("-" * 50)

inputs = sum_tokenizer(text_to_summarize, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = sum_model.generate(inputs["input_ids"], max_length=50, min_length=20, num_beams=4)
summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Generated Summary:\n", summary)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Original Text: Encoder-decoder architectures, originally developed for statistical machine translation, have become a cornerstone in natural language processing (NLP). 
An encoder processes the input data (like text or an image) and compresses the information into a context representation. 
Following this, the decoder utilizes the context representation to generate the target output, sequence by sequence. 
This configuration has proven universally effective across disparate sequence-to-sequence tasks, encompassing translation, summarization, 
and questioning answering. Recent variations introduced the self-attention mechanism and entirely discarded RNN architectures 
in favor of Transformer models, yielding vastly improved state-of-the-art results across varied benchmarks.
--------------------------------------------------
Generated Summary:
 An encoder processes the input data (like text or an image) and compresses the information into a context representation. The decoder utilizes th

In [13]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading translation model...")
trans_tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
trans_mdl = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es")

print("Loading summarization model...")
sum_tok = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_mdl = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

print("Models loaded!")

def translate_text(text):
    if not text.strip():
        return "Please provide text to translate."
    try:
        inputs = trans_tok(text, return_tensors="pt", truncation=True, max_length=512)
        translated = trans_mdl.generate(**inputs, max_length=512)
        return trans_tok.decode(translated[0], skip_special_tokens=True)
    except Exception as e:
        return f"Error: {str(e)}"

def summarize_text(text):
    if not text.strip():
        return "Please provide text to summarize."
    try:
        inputs = sum_tok(text, return_tensors="pt", truncation=True, max_length=1024)
        ids = sum_mdl.generate(inputs["input_ids"], max_length=50, min_length=10, num_beams=4)
        return sum_tok.decode(ids[0], skip_special_tokens=True)
    except Exception as e:
        return f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("# 🚀 Encoder-Decoder Architecture Dashboard")
    gr.Markdown("Interact with Encoder-Decoder models for **Machine Translation** and **Text Summarization**.")

    with gr.Tab("🇬🇧 English → 🇪🇸 Spanish Translation"):
        with gr.Row():
            with gr.Column():
                trans_input  = gr.Textbox(lines=5, label="Input English Text", placeholder="Enter English text here...")
                trans_btn    = gr.Button("Translate to Spanish", variant="primary")
            with gr.Column():
                trans_output = gr.Textbox(lines=5, label="Spanish Output")
        trans_btn.click(translate_text, inputs=trans_input, outputs=trans_output)

    with gr.Tab("📝 Text Summarization"):
        with gr.Row():
            with gr.Column():
                sum_input  = gr.Textbox(lines=8, label="Input Long Text", placeholder="Enter a long paragraph here...")
                sum_btn    = gr.Button("Summarize", variant="primary")
            with gr.Column():
                sum_output = gr.Textbox(lines=8, label="Summarized Output")
        sum_btn.click(summarize_text, inputs=sum_input, outputs=sum_output)

print("Launching Gradio dashboard...")
dashboard.launch(share=True, inline=True)

Loading translation model...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading summarization model...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Models loaded!
Launching Gradio dashboard...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b411f21c9794654fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
